# TC3-45 – Analisar nulos, duplicidades e outliers

**Responsável:** Mauricio Cruz

**Sprint:** Fase 3

**Jira:** TC3-45

## Objetivo

Quantificar e investigar valores nulos, registros duplicados e possíveis outliers, registrando impactos e necessidades de tratamento.


## Inicialização independente

Esta célula reproduz a configuração compartilhada do notebook principal. Ela prepara ambiente, caminhos e funções comuns sem alterar a lógica canônica desta etapa.


In [1]:
# =============================================================================
# CONFIGURAÇÕES PARA RODAR O NOTEBOOK
# Execute esta célula primeiro. Esta política não altera a metodologia do fluxo.
# =============================================================================
import os
import sys
import gc
import ctypes
import polars as pl
from datetime import datetime, timezone

# Opções do usuário
PERFIL_RECURSOS = "auto"       # auto | baixo | medio | normal | alto
LIMITE_RAM_PERCENTUAL = None    # None = automático; ou inteiro entre 50 e 90
MAX_THREADS = None              # None = automático; ou inteiro >= 1
USAR_GPU = os.getenv("TC3_USAR_GPU", "1").strip().lower() in {"1", "true", "sim", "yes"}                 # requer WSL2 + RAPIDS/cuML + kernel tc3-gpu

REUTILIZAR_ARTEFATOS = False  # False reconstrói; True aceita caches válidos
SALVAR_ARTEFATOS = True       # False usa somente área temporária da sessão


# Protocolo metodologico compartilhado por todas as fases e modelos.
SEED_GLOBAL = 42
EXECUTAR_SHAP = os.getenv("TC3_EXECUTAR_SHAP", "0").strip().lower() in {"1", "true", "sim", "yes"}
AMOSTRA_SHAP = max(20, int(os.getenv("TC3_AMOSTRA_SHAP", "100")))
AMOSTRA_BACKGROUND_SHAP = max(20, int(os.getenv("TC3_BACKGROUND_SHAP", "50")))
CHAVE_GRUPO_GLOBAL = "chave_escola_ano"
N_SPLITS_CV_GLOBAL = 3
N_ITER_BUSCA_GLOBAL = 10
LIMIAR_CLASSIFICACAO_GLOBAL = 0.50
REPETICOES_PERMUTACAO_GLOBAL = 5
REPETICOES_BOOTSTRAP_GLOBAL = 200

N_JOBS_MODELAGEM_GLOBAL = 1
N_SPLITS_HOLDOUT_GLOBAL = 1
SCORING_PRINCIPAL_GLOBAL = "roc_auc"
VERBOSE_BUSCA_GLOBAL = 2
ERRO_BUSCA_GLOBAL = "raise"
RETORNAR_SCORE_TREINO_GLOBAL = True

# Contrato comum de pre-processamento dos modelos.
IMPUTACAO_NUMERICA_GLOBAL = "median"
IMPUTACAO_CATEGORICA_GLOBAL = "most_frequent"
ONE_HOT_DESCONHECIDO_GLOBAL = "ignore"
ONE_HOT_SPARSO_GLOBAL = False  # matriz densa: evita fallback dos modelos para CPU
SPARSE_THRESHOLD_GLOBAL = 0.0
ESCALONADOR_CENTRALIZAR_GLOBAL = False


# Percentuais de amostragem (0 < valor <= 1). Alterar aqui afeta todo o notebook.
# O teste final permanece integral (100%); as amostras menores servem apenas ao
# desenvolvimento, comparacao de modelos e interpretabilidade.
FRACOES_AMOSTRAGEM = {
    "eda_correlacao": 0.05,
    "validacao_preparacao": 0.02,
    "fase5_modelos": 0.10,
    "fase5_knn": 0.02,  # KNN tem custo de memoria e predicao maior.
    "fase6_desenvolvimento": 0.12,
    "fase6_validacao_interna": 0.25,
    "fase7_desenvolvimento": 1.00,
    "fase7_teste": 1.00,
    "interpretacao": 0.035,
    "bootstrap": 0.17,
}

for _nome, _fracao in FRACOES_AMOSTRAGEM.items():
    if not 0 < _fracao <= 1:
        raise ValueError(f"Fracao invalida em FRACOES_AMOSTRAGEM[{_nome!r}]: {_fracao}")


def tamanho_por_fracao(total, fracao):
    """Converte uma proporcao em quantidade reproduzivel, sem exceder o total."""
    if total < 1:
        raise ValueError("Nao e possivel amostrar uma particao vazia.")
    return min(total, max(1, round(total * fracao)))


def materializar_split_percentual(dados, contrato, nome_split, fracao, seed, materializador):
    """Materializa uma fracao de um split e informa claramente o universo usado."""
    import polars as pl

    coluna_split = contrato["controle_split"]
    total = (
        dados.filter(pl.col(coluna_split) == nome_split)
        .select(pl.len().alias("total"))
        .collect()
        .item()
    )
    quantidade = tamanho_por_fracao(total, fracao)
    X, y = materializador(dados, contrato, nome_split, limit=quantidade, seed=seed)
    print(f"{nome_split}: {quantidade:,}/{total:,} registros ({fracao:.1%})")
    return X, y


def _estado_memoria_sistema():
    """Retorna RAM total e disponível em bytes."""
    valores = {}
    try:
        with open("/proc/meminfo", encoding="utf-8") as arquivo:
            for linha in arquivo:
                chave, valor = linha.split(":", 1)
                valores[chave] = int(valor.strip().split()[0]) * 1024
        return valores["MemTotal"], valores.get("MemAvailable", valores.get("MemFree", 0))
    except (OSError, KeyError, ValueError):
        try:
            return (
                os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"),
                os.sysconf("SC_AVPHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"),
            )
        except (AttributeError, OSError, ValueError):
            return None, None


RAM_TOTAL_BYTES, RAM_DISPONIVEL_BYTES = _estado_memoria_sistema()
RAM_TOTAL_GB = RAM_TOTAL_BYTES / 1024**3 if RAM_TOTAL_BYTES else 0.0
CPUS_LOGICAS = os.cpu_count() or 1

if RAM_TOTAL_GB and RAM_TOTAL_GB <= 4.5:
    _perfil_detectado = "baixo"
elif RAM_TOTAL_GB and RAM_TOTAL_GB <= 8.5:
    _perfil_detectado = "medio"
elif RAM_TOTAL_GB and RAM_TOTAL_GB <= 16.5:
    _perfil_detectado = "normal"
else:
    _perfil_detectado = "alto"

_perfis = {
    "baixo": (60, 1),
    "medio": (70, min(2, CPUS_LOGICAS)),
    "normal": (75, min(4, CPUS_LOGICAS)),
    "alto": (80, min(8, max(1, CPUS_LOGICAS // 2))),
}
PERFIL_ATIVO = _perfil_detectado if PERFIL_RECURSOS == "auto" else PERFIL_RECURSOS.lower()
if PERFIL_ATIVO not in _perfis:
    raise ValueError("PERFIL_RECURSOS deve ser auto, baixo, medio, normal ou alto.")

_limite_perfil, _threads_perfil = _perfis[PERFIL_ATIVO]
LIMITE_RAM_ATIVO = _limite_perfil if LIMITE_RAM_PERCENTUAL is None else int(LIMITE_RAM_PERCENTUAL)
THREADS_MAXIMAS = _threads_perfil if MAX_THREADS is None else int(MAX_THREADS)
if not 50 <= LIMITE_RAM_ATIVO <= 90:
    raise ValueError("LIMITE_RAM_PERCENTUAL deve ficar entre 50 e 90.")
if THREADS_MAXIMAS < 1:
    raise ValueError("MAX_THREADS deve ser pelo menos 1.")
THREADS_MAXIMAS = min(THREADS_MAXIMAS, CPUS_LOGICAS)

# Definido antes de NumPy, Polars, scikit-learn e cuML serem importados.
for _variavel, _valor in {
    "OMP_NUM_THREADS": str(THREADS_MAXIMAS),
    "OPENBLAS_NUM_THREADS": str(THREADS_MAXIMAS),
    "MKL_NUM_THREADS": str(THREADS_MAXIMAS),
    "NUMEXPR_NUM_THREADS": str(THREADS_MAXIMAS),
    "VECLIB_MAXIMUM_THREADS": str(THREADS_MAXIMAS),
    "POLARS_MAX_THREADS": str(THREADS_MAXIMAS),
    "MALLOC_ARENA_MAX": "2",
}.items():
    os.environ[_variavel] = _valor

HISTORICO_RECURSOS = []
_estado_recursos = {"nivel": None, "threads": None, "contexto_threads": None}


def _aplicar_threads(limite):
    """Limita pools BLAS/OpenMP já carregados; Polars usa o limite inicial."""
    limite = max(1, min(int(limite), THREADS_MAXIMAS))
    if _estado_recursos["threads"] == limite:
        return
    anterior = _estado_recursos.get("contexto_threads")
    if anterior is not None:
        try:
            anterior.__exit__(None, None, None)
        except Exception:
            pass
    try:
        from threadpoolctl import threadpool_limits
        contexto = threadpool_limits(limits=limite)
        contexto.__enter__()
    except Exception:
        contexto = None
    _estado_recursos["contexto_threads"] = contexto
    _estado_recursos["threads"] = limite


def _liberar_memoria_global(agressiva=False):
    """Libera caches transitórios sem apagar variáveis usadas pelo fluxo."""
    shell = get_ipython()
    if shell is not None:
        for nome in ("_", "__", "___"):
            shell.user_ns[nome] = None
    gc.collect()
    modulo_cupy = sys.modules.get("cupy")
    if modulo_cupy is not None:
        try:
            modulo_cupy.get_default_memory_pool().free_all_blocks()
            modulo_cupy.get_default_pinned_memory_pool().free_all_blocks()
        except Exception:
            pass
    if agressiva:
        modulo_plt = sys.modules.get("matplotlib.pyplot")
        if modulo_plt is not None:
            try:
                modulo_plt.close("all")
            except Exception:
                pass
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except (OSError, AttributeError):
            pass


def _gerenciar_recursos(*_args, **_kwargs):
    """Reduz paralelismo progressivamente e nunca interrompe uma célula."""
    total, disponivel = _estado_memoria_sistema()
    if not total:
        return
    uso = 100.0 * (total - disponivel) / total
    margem = LIMITE_RAM_ATIVO - uso
    if margem > 15:
        nivel, threads, limpar = "normal", THREADS_MAXIMAS, False
    elif margem > 7:
        nivel, threads, limpar = "moderado", max(1, THREADS_MAXIMAS // 2), True
    else:
        nivel, threads, limpar = "conservador", 1, True

    _aplicar_threads(threads)
    if limpar:
        _liberar_memoria_global(agressiva=margem <= 0)

    if nivel != _estado_recursos["nivel"]:
        registro = {
            "instante_utc": datetime.now(timezone.utc).isoformat(),
            "nivel": nivel,
            "threads": threads,
            "ram_usada_percentual": round(uso, 1),
            "ram_disponivel_gb": round(disponivel / 1024**3, 2),
        }
        HISTORICO_RECURSOS.append(registro)
        print(
            f"[recursos] nível={nivel}; threads={threads}; "
            f"RAM usada={uso:.1f}%; livre={disponivel / 1024**3:.1f} GB."
        )
        _estado_recursos["nivel"] = nivel


ipython = get_ipython()
if ipython is not None:
    # Impede callbacks duplicados ao reexecutar a configuração.
    for _atributo in ("_tc3_memory_guard", "_tc3_resource_guard"):
        _guard_anterior = getattr(ipython, _atributo, None)
        if _guard_anterior is not None:
            for _evento, _funcao in _guard_anterior:
                try:
                    ipython.events.unregister(_evento, _funcao)
                except (ValueError, KeyError):
                    pass
    _callbacks_recursos = [
        ("pre_run_cell", _gerenciar_recursos),
        ("post_run_cell", _gerenciar_recursos),
    ]
    for _evento, _funcao in _callbacks_recursos:
        ipython.events.register(_evento, _funcao)
    ipython._tc3_resource_guard = _callbacks_recursos

_gerenciar_recursos()
print(
    "Configuração automática ativa: "
    f"perfil={PERFIL_ATIVO}; RAM={RAM_TOTAL_GB:.1f} GB; CPUs={CPUS_LOGICAS}; "
    f"limite preventivo={LIMITE_RAM_ATIVO}%; threads máximas={THREADS_MAXIMAS}."
)

# Backend GPU: ativado antes de qualquer importação do scikit-learn.
cuda_conda = os.path.join(sys.prefix, "targets", "x86_64-linux")
if os.path.exists(os.path.join(cuda_conda, "include", "cuda.h")):
    os.environ.setdefault("CUDA_PATH", cuda_conda)
if USAR_GPU:
    try:
        from cuml.accel import install
        install(log_level="info")
        import cupy as cp
        dispositivo = cp.cuda.runtime.getDeviceProperties(0)
        nome_gpu = dispositivo["name"].decode() if isinstance(dispositivo["name"], bytes) else str(dispositivo["name"])
        memoria_gpu_gb = cp.cuda.runtime.memGetInfo()[1] / 1024**3
        BACKEND_ML = f"RAPIDS cuML — {nome_gpu} ({memoria_gpu_gb:.1f} GB de VRAM)"
    except Exception as exc:
        raise RuntimeError("GPU/RAPIDS indisponível. Abra no WSL2 e selecione o kernel tc3-gpu.") from exc
else:
    BACKEND_ML = "scikit-learn CPU"
print("Backend de Machine Learning:", BACKEND_ML)

from pathlib import Path
import hashlib
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

def localizar_raiz(inicio: Path) -> Path:
    for candidato in [inicio, *inicio.parents]:
        if (candidato / "dados").is_dir() and (candidato / "notebooks").is_dir():
            return candidato
    raise FileNotFoundError("Execute este notebook dentro do repositório tech_challenge_3.")

PROJECT_ROOT = localizar_raiz(Path.cwd().resolve())
print("Raiz do projeto:", PROJECT_ROOT)


# Contrato único de diretórios. Fontes sempre são lidas da área imutável.
import tempfile

FONTES_ORIGINAIS_DIR = PROJECT_ROOT / "dados" / "fontes_originais"
_TEMP_BASE = PROJECT_ROOT / ".execucao_temporaria"
_TEMP_BASE.mkdir(parents=True, exist_ok=True)
if SALVAR_ARTEFATOS:
    RAIZ_ARTEFATOS = PROJECT_ROOT
    _DIRETORIO_TEMPORARIO = None
else:
    _DIRETORIO_TEMPORARIO = tempfile.TemporaryDirectory(prefix="tc3_", dir=_TEMP_BASE)
    RAIZ_ARTEFATOS = Path(_DIRETORIO_TEMPORARIO.name)

DADOS_PROCESSADOS_DIR = RAIZ_ARTEFATOS / "dados" / "dados_processados"
DADOS_ANALITICOS_DIR = RAIZ_ARTEFATOS / "dados" / "dados_analiticos"
RESULTADOS_DIR = RAIZ_ARTEFATOS / "resultados"
RESULTADOS_FIGURAS_DIR = RESULTADOS_DIR / "figuras"
RESULTADOS_TABELAS_DIR = RESULTADOS_DIR / "tabelas"
RESULTADOS_METRICAS_DIR = RESULTADOS_DIR / "metricas"
RESULTADOS_RELATORIOS_DIR = RESULTADOS_DIR / "relatorios"
MODELOS_DIR = RAIZ_ARTEFATOS / "modelos"

for _diretorio_saida in (
    DADOS_PROCESSADOS_DIR, DADOS_ANALITICOS_DIR, RESULTADOS_FIGURAS_DIR,
    RESULTADOS_TABELAS_DIR, RESULTADOS_METRICAS_DIR,
    RESULTADOS_RELATORIOS_DIR, MODELOS_DIR,
):
    _diretorio_saida.mkdir(parents=True, exist_ok=True)

def deve_recalcular(caminho):
    """True quando o cache deve ser ignorado ou ainda não existe."""
    return not REUTILIZAR_ARTEFATOS or not Path(caminho).exists()


def publicar_parquet_particionado(frame, destino, *, particionar_por):
    """Publica partições sequencialmente e troca o diretório de forma atômica.

    O ``write_parquet(partition_by=...)`` do Polars prepara várias partições em
    paralelo e pode duplicar o pico de RAM de bases já materializadas. Esta
    implementação mantém apenas uma partição adicional em memória por vez.
    """
    import gc
    import shutil

    destino = Path(destino)
    temporario = destino.with_name(destino.name + "__publicacao_tmp")
    anterior = destino.with_name(destino.name + "__publicacao_anterior")
    for caminho in (temporario, anterior):
        if caminho.exists():
            shutil.rmtree(caminho)
    temporario.mkdir(parents=True, exist_ok=True)

    lazy = frame if isinstance(frame, pl.LazyFrame) else frame.lazy()
    schema = lazy.collect_schema()
    if particionar_por not in schema.names():
        raise ValueError(f"Coluna de particionamento ausente: {particionar_por}")

    valores = (
        lazy.select(pl.col(particionar_por).unique())
        .collect(engine="streaming")
        .get_column(particionar_por)
        .to_list()
    )
    valores = sorted(valores, key=lambda valor: (valor is None, str(valor)))
    if not valores:
        raise RuntimeError("Não há valores para particionar a publicação.")

    arquivos_novos = []
    for indice, valor in enumerate(valores):
        valor_hive = "__HIVE_DEFAULT_PARTITION__" if valor is None else str(valor)
        if any(separador in valor_hive for separador in ("/", "\\")):
            raise ValueError(f"Valor de partição inválido: {valor_hive!r}")
        diretorio_particao = temporario / f"{particionar_por}={valor_hive}"
        diretorio_particao.mkdir(parents=True, exist_ok=True)
        arquivo = diretorio_particao / f"part-{indice:05d}.parquet"
        predicado = pl.col(particionar_por).is_null() if valor is None else pl.col(particionar_por) == pl.lit(valor)
        lazy.filter(predicado).sink_parquet(
            arquivo,
            compression="zstd",
            statistics=True,
            row_group_size=100_000,
            mkdir=True,
            engine="streaming",
        )
        if not arquivo.is_file() or arquivo.stat().st_size == 0:
            raise RuntimeError(f"Partição não foi gravada corretamente: {arquivo}")
        arquivos_novos.append(arquivo)
        gc.collect()

    if destino.exists():
        destino.replace(anterior)
    try:
        try:
            temporario.replace(destino)
        except OSError:
            # Volumes Docker montados do Windows podem bloquear rename de diretórios.
            # O fallback copia a publicação pronta e remove a área temporária.
            shutil.copytree(temporario, destino)
            shutil.rmtree(temporario)
    except Exception:
        if destino.exists():
            shutil.rmtree(destino)
        if anterior.exists():
            anterior.replace(destino)
        raise
    if anterior.exists():
        shutil.rmtree(anterior)
    return len(arquivos_novos)


def salvar_figura_eda(figura, nome):
    """Exporta uma figura exploratória sem alterar sua exibição no notebook."""
    destino = RESULTADOS_FIGURAS_DIR / "fluxo_exploratorio" / nome
    destino.parent.mkdir(parents=True, exist_ok=True)
    figura.savefig(destino, dpi=180, bbox_inches="tight")
    return destino

print("Fontes originais:", FONTES_ORIGINAIS_DIR)
print("Destino dos artefatos:", RAIZ_ARTEFATOS)
print("Reutilizar artefatos:", REUTILIZAR_ARTEFATOS, "| Salvar:", SALVAR_ARTEFATOS)

# Imports compartilhados que, no fluxo completo, são carregados por etapas anteriores.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


[recursos] nível=normal; threads=4; RAM usada=28.8%; livre=11.0 GB.
Configuração automática ativa: perfil=normal; RAM=15.5 GB; CPUs=20; limite preventivo=75%; threads máximas=4.
[cuml.accel] Accelerator installed.
Backend de Machine Learning: RAPIDS cuML — NVIDIA GeForce RTX 3050 6GB Laptop GPU (6.0 GB de VRAM)
Raiz do projeto: /workspace
Fontes originais: /workspace/dados/fontes_originais
Destino dos artefatos: /workspace
Reutilizar artefatos: False | Salvar: True


In [2]:
import polars as pl
from pathlib import Path
import matplotlib.pyplot as plt

pl.Config.set_tbl_rows(-1)


polars.config.Config

### Carregar base de dados


In [3]:
# Recarregar a base com o caminho correto
BASE_PATH = DADOS_PROCESSADOS_DIR / "consolidado" / "base_alunos_municipios_por_uf"

df = pl.scan_parquet(
    str(BASE_PATH / "sigla_uf=*" / "*.parquet"),
    hive_partitioning=True
)

# Atualizar o LazyFrame de análise
df_analise = df

In [4]:
# Entrada mínima herdada do TC3-44: mesma amostragem determinística do fluxo principal.
FRACAO_EDA_DISTRIBUICOES = FRACOES_AMOSTRAGEM["eda_correlacao"]
if isinstance(df_analise, pl.LazyFrame):
    limite_hash = max(1, round(FRACAO_EDA_DISTRIBUICOES * 10_000))
    df_sample = (
        df_analise
        .filter((pl.col("id_aluno").hash(seed=SEED_GLOBAL) % 10_000) < limite_hash)
        .collect(engine="streaming")
    )
else:
    df_sample = df_analise.sample(
        fraction=FRACAO_EDA_DISTRIBUICOES,
        seed=SEED_GLOBAL,
        shuffle=True,
    )
print(f"Amostra EDA: {df_sample.height:,} registros ({FRACAO_EDA_DISTRIBUICOES:.1%}).")


Amostra EDA: 192,850 registros (5.0%).


In [5]:
# Mantém a base como LazyFrame. Materializar milhões de linhas aqui duplicaria
# a memória já usada pela amostra exploratória da seção anterior.
schema_analise = df_analise.collect_schema()
total_registros_analise = (
    df_analise.select(pl.len().alias("total")).collect(engine="streaming").item()
)
print(f"Total de registros: {total_registros_analise:,}")
print(f"Total de colunas  : {len(schema_analise)}")


Total de registros: 3,867,999
Total de colunas  : 49


In [6]:
# ============================================================
# 1. VALORES NULOS 
# ============================================================
print("\n" + "="*70)
print("1. VALORES NULOS – CONSOLIDAÇÃO")
print("="*70)

null_summary = (
    df_analise
    .select([pl.col(c).null_count().alias(c) for c in schema_analise.names()])
    .collect(engine="streaming")
    .transpose(include_header=True, header_name="coluna", column_names=["nulos"])
    .with_columns(
        (pl.col("nulos") / total_registros_analise * 100).round(2).alias("pct_nulos")
    )
    .sort("pct_nulos", descending=True)
)

print(null_summary.filter(pl.col("pct_nulos") > 0))

print(f"\nColunas com nulos     : {null_summary.filter(pl.col('nulos') > 0).height}")
print(f"Colunas 100% preenchidas: {null_summary.filter(pl.col('nulos') == 0).height}")



1. VALORES NULOS – CONSOLIDAÇÃO
shape: (17, 3)
┌─────────────────────────────────┬─────────┬───────────┐
│ coluna                          ┆ nulos   ┆ pct_nulos │
│ ---                             ┆ ---     ┆ ---       │
│ str                             ┆ u32     ┆ f64       │
╞═════════════════════════════════╪═════════╪═══════════╡
│ taxa_alfabetizacao_municipio_a… ┆ 1791646 ┆ 46.32     │
│ media_portugues_municipio_ano_… ┆ 1791646 ┆ 46.32     │
│ proficiencia                    ┆ 513338  ┆ 13.27     │
│ peso_aluno                      ┆ 513338  ┆ 13.27     │
│ valor_receitas                  ┆ 34500   ┆ 0.89      │
│ valor_fundeb                    ┆ 33712   ┆ 0.87      │
│ qtd_escolas_inse                ┆ 2330    ┆ 0.06      │
│ qtd_alunos_inse                 ┆ 2330    ┆ 0.06      │
│ media_inse                      ┆ 2330    ┆ 0.06      │
│ pc_nivel_1                      ┆ 2330    ┆ 0.06      │
│ pc_nivel_2                      ┆ 2330    ┆ 0.06      │
│ pc_nivel_3            

In [7]:
# ============================================================
# 2. REGISTROS DUPLICADOS
# ============================================================
print("\n" + "="*70)
print("2. REGISTROS DUPLICADOS")
print("="*70)

# 2.1 Duplicatas de linha inteira. O hash evita criar uma segunda cópia
# completa da tabela, como ocorria com df_analise.unique().
duplicatas_linha = (
    df_analise
    .select(pl.struct(pl.all()).hash(seed=SEED_GLOBAL).alias("hash_linha"))
    .group_by("hash_linha").len()
    .filter(pl.col("len") > 1)
    .select((pl.col("len") - 1).sum().fill_null(0).alias("duplicatas"))
    .collect(engine="streaming").item()
)
print(f"Duplicatas de linha inteira: {duplicatas_linha:,}")

# 2.2 Duplicatas pela chave de aluno (mais importante)
if "id_aluno" in schema_analise.names():
    dup_aluno_lf = (
        df_analise
        .group_by("id_aluno")
        .len().rename({"len": "qtd"})
        .filter(pl.col("qtd") > 1)
    )
    resumo_dup_aluno = dup_aluno_lf.select(pl.len().alias("grupos")).collect(engine="streaming").item()
    print(f"Alunos com mais de um registro: {resumo_dup_aluno:,}")
    if resumo_dup_aluno > 0:
        print(dup_aluno_lf.sort("qtd", descending=True).head(10).collect(engine="streaming"))
else:
    print("Coluna 'id_aluno' não encontrada.")

# 2.3 Duplicatas por id_aluno + ano_referencia (se fizer sentido)
if all(c in schema_analise.names() for c in ["id_aluno", "ano_referencia"]):
    dup_aluno_ano_lf = (
        df_analise
        .group_by(["id_aluno", "ano_referencia"])
        .len().rename({"len": "qtd"})
        .filter(pl.col("qtd") > 1)
    )
    resumo_dup_aluno_ano = dup_aluno_ano_lf.select(pl.len().alias("grupos")).collect(engine="streaming").item()
    print(f"Duplicatas por (id_aluno + ano_referencia): {resumo_dup_aluno_ano:,}")
    if resumo_dup_aluno_ano > 0:
        print(dup_aluno_ano_lf.sort("qtd", descending=True).head(5).collect(engine="streaming"))



2. REGISTROS DUPLICADOS
Duplicatas de linha inteira: 0
Alunos com mais de um registro: 1,515,671
shape: (10, 2)
┌──────────┬─────┐
│ id_aluno ┆ qtd │
│ ---      ┆ --- │
│ str      ┆ u32 │
╞══════════╪═════╡
│ 31062761 ┆ 2   │
│ 31126551 ┆ 2   │
│ 31136591 ┆ 2   │
│ 31035226 ┆ 2   │
│ 31055219 ┆ 2   │
│ 31147781 ┆ 2   │
│ 31201122 ┆ 2   │
│ 31197912 ┆ 2   │
│ 31125536 ┆ 2   │
│ 31152241 ┆ 2   │
└──────────┴─────┘
Duplicatas por (id_aluno + ano_referencia): 0


In [8]:

# ============================================================
# 3. POSSÍVEIS OUTLIERS (método IQR)
# ============================================================
print("\n" + "="*70)
print("3. POSSÍVEIS OUTLIERS (Método IQR)")
print("="*70)

# Features numéricas relevantes para análise de outliers
features_outliers = [
    c for c in [
        "proficiencia", "peso_aluno",
        "valor_bolsa_familia", "valor_fundeb", "valor_receitas",
        "valor_pib", "valor_pib_per_capita",
        "media_inse",
        "taxa_alfabetizacao_municipio_ano_anterior",
        "media_portugues_municipio_ano_anterior",
        "qtd_alunos_inse", "qt_mat_bas", "qt_doc_bas", "qt_tur_bas"
    ] if c in schema_analise.names()
]

# Quartis exploratórios na amostra determinística; contagens finais em 100%
# da base. Assim, nenhuma coluna completa é convertida em Series na memória.
outlier_results = []
limites_outliers = {}
for col in features_outliers:
    estatisticas = df_sample.select(
        pl.col(col).count().alias("n"),
        pl.col(col).quantile(0.25).alias("q1"),
        pl.col(col).quantile(0.75).alias("q3"),
        pl.col(col).min().alias("min"),
        pl.col(col).max().alias("max"),
    ).row(0, named=True)
    if estatisticas["n"] < 100 or estatisticas["q1"] is None:
        continue
    q1, q3 = estatisticas["q1"], estatisticas["q3"]
    iqr = q3 - q1
    limites_outliers[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr, estatisticas)

expressoes_outliers = []
for col, (limite_inf, limite_sup, _) in limites_outliers.items():
    expressoes_outliers.extend([
        pl.col(col).count().alias(f"{col}__n"),
        ((pl.col(col) < limite_inf) | (pl.col(col) > limite_sup)).sum().alias(f"{col}__outliers"),
    ])
contagens_outliers = (
    df_analise.select(expressoes_outliers).collect(engine="streaming").row(0, named=True)
    if expressoes_outliers else {}
)
for col, (limite_inf, limite_sup, estatisticas) in limites_outliers.items():
    n_validos = contagens_outliers[f"{col}__n"]
    n_outliers = contagens_outliers[f"{col}__outliers"] or 0
    q1, q3 = estatisticas["q1"], estatisticas["q3"]
    outlier_results.append({
        "coluna": col, "q1": round(q1, 2), "q3": round(q3, 2),
        "iqr": round(q3 - q1, 2), "limite_inf": round(limite_inf, 2),
        "limite_sup": round(limite_sup, 2), "n_outliers": n_outliers,
        "pct_outliers": round(100 * n_outliers / n_validos, 2) if n_validos else 0.0,
        "min": round(estatisticas["min"], 2), "max": round(estatisticas["max"], 2),
    })

df_outliers = pl.DataFrame(outlier_results).sort("pct_outliers", descending=True)
print(df_outliers)



3. POSSÍVEIS OUTLIERS (Método IQR)
shape: (14, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ coluna    ┆ q1        ┆ q3        ┆ iqr       ┆ … ┆ n_outlier ┆ pct_outli ┆ min       ┆ max      │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ s         ┆ ers       ┆ ---       ┆ ---      │
│ str       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ ---       ┆ ---       ┆ f64       ┆ f64      │
│           ┆           ┆           ┆           ┆   ┆ i64       ┆ f64       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ valor_fun ┆ -1.2144e7 ┆ 6.9188e8  ┆ 7.0403e8  ┆ … ┆ 994607    ┆ 25.94     ┆ -4.5338e1 ┆ 3.5589e1 │
│ deb       ┆           ┆           ┆           ┆   ┆           ┆           ┆ 0         ┆ 0        │
│ valor_pib ┆ 7.83841e8 ┆ 2.2791e10 ┆ 2.2007e10 ┆ … ┆ 604083    ┆ 15.62     ┆ 2.1017664 ┆ 1.0668e1 │
│           ┆           ┆           ┆  

## Resultados interpretados

### 1. Valores Nulos

- Já consolidado anteriormente.
- 33 colunas com nulos | 16 colunas 100% preenchidas.
- Bloco crítico (~54,85%) continua sendo o principal problema (INSE + Censo + PIB)


### 2. Registros Duplicados

| Tipo de duplicata | Quantidade | Interpretação |
|---|---:|---|
| Linha inteira | **0** | Nenhuma linha 100% idêntica. |
| `id_aluno` (qualquer ano) | **1.515.671** | Alunos que aparecem em mais de um ano (2023 e 2024). |
| `id_aluno` + `ano_referencia` | **0** | Não há duplicatas reais dentro do mesmo ano. |

Conclusão:

Não existem duplicatas problemáticas. A repetição de id_aluno é esperada porque a base contém dois anos. Cada aluno aparece no máximo uma vez por ano.
Impacto: Nenhum.

Tratamento necessário: Nenhum (manter como está).


### 3. Outliers (Método IQR) – Principais achados

| Feature | % Outliers | Min | Max | Observação importante | Tratamento recomendado |
|---|---:|---:|---:|---|---|
| `valor_fundeb` | **25,73%** | **-45,3 bi** | 35,6 bi | **Valores negativos** + extrema amplitude | Investigar a origem dos negativos + `log1p` ou tratamento especial. |
| `valor_bolsa_familia` | 14,90% | 41.280 | 5,24 bi | Forte assimetria | `log1p`. |
| `valor_receitas` | 14,39% | 8,4 mi | 257,5 bi | Forte assimetria | `log1p`. |
| `qt_mat_bas` / `qt_doc_bas` | 13,34% | — | — | Municípios grandes (capitais) | Avaliar se os valores são legítimos. |
| `valor_pib` | 13,18% | — | — | Mesmo padrão financeiro | `log1p`. |
| `qtd_alunos_inse` | 12,89% | 15 | 149.968 | Municípios grandes | Avaliar se os valores são legítimos. |
| `media_inse` | **0,00%** | 3,50 | 5,96 | Excelente comportamento | Nenhum tratamento necessário. |


Destaque crítico:

valor_fundeb possui valores negativos (mínimo ≈ -45 bilhões). Isso é semanticamente estranho para um indicador de transferência de recursos e precisa ser investigado na fonte.


### 4. Documentação Final do Requisito

#### Quantificação e Investigação de Valores Nulos, Duplicados e Outliers

#### 4.1 Valores Nulos
- **33 colunas** com valores nulos (67% das features).
- **16 colunas** 100% preenchidas.
- Bloco crítico (INSE + Censo Escolar + PIB + infraestrutura): **~54,85%** de nulos.
- Causa raiz: incompatibilidade temporal nos joins (bases externas de 2023 vs alunos de 2024).
- **Impacto:** Alto – perda de informação socioeconômica e de infraestrutura para mais da metade da base.
- **Tratamento necessário:** 
  1. Corrigir joins com fallback para o ano anterior (prioritário).
  2. Criar flags de missing.
  3. Imputação (mediana ou por UF) nas features restantes.

#### 4.2 Registros Duplicados
- Duplicatas de linha inteira: **0**.
- Duplicatas por (`id_aluno` + `ano_referencia`): **0**.
- Alunos presentes em mais de um ano: 1.515.671 (comportamento esperado).
- **Impacto:** Nenhum.
- **Tratamento necessário:** Nenhum.

#### 4.3 Possíveis Outliers (Método IQR)
- Features financeiras (`valor_fundeb`, `valor_bolsa_familia`, `valor_receitas`, `valor_pib`) apresentam entre **13% e 26%** de outliers.
- `valor_fundeb` possui valores **negativos** (mínimo ≈ -45,3 bilhões) → requer investigação.
- `media_inse` e features de desempenho educacional não apresentam outliers relevantes.
- Contagens escolares (`qt_mat_bas`, `qt_doc_bas`, `qtd_alunos_inse`) têm outliers legítimos (municípios grandes).

**Impacto:**  
Alto nas variáveis financeiras (podem distorcer modelos lineares e baseados em distância).  
Baixo nas features educacionais.

**Tratamentos recomendados:**
1. Investigar e tratar valores negativos de `valor_fundeb`.
2. Aplicar transformação `log1p` nas variáveis financeiras e em `peso_aluno`.
3. Avaliar winsorização leve após a transformação log (opcional).
4. Manter `media_inse` e `proficiencia` na escala original.

#### 4.4 Resumo de Prioridades de Tratamento

| Prioridade | Ação                                      | Justificativa                          |
|------------|-------------------------------------------|----------------------------------------|
| 1          | Corrigir joins (fallback de ano)          | Maior impacto na qualidade dos dados   |
| 2          | Investigar valores negativos de Fundeb    | Inconsistência semântica               |
| 3          | Transformação log1p (financeiras)         | Reduz assimetria e efeito de outliers  |
| 4          | Flags de missing + imputação              | Preservar informação e sinalizar falha |
| 5          | Nenhuma ação em duplicatas                | Não há duplicatas problemáticas        |


### Causa dos valores negativos em valor_fundeb

Os valores negativos não são erro de dado. Eles refletem a mecânica real do Fundeb.

#### Como o Fundeb funciona (resumo)

1. Cada município (e estado) contribui com 20% de uma cesta de impostos e transferências (ICMS, FPM, IPVA, etc.) para o fundo estadual do Fundeb.
2. Esses recursos são redistribuídos entre as redes de ensino do estado com base no número de matrículas (ponderadas).
3. A União ainda complementa o fundo (VAAF, VAAT e VAAR).

#### Por que aparece valor negativo?
O campo total_valor da base que você está usando representa, na prática, o resultado líquido do município no Fundeb:

Valor líquido ≈ (recursos recebidos do Fundeb) − (recursos que o município contribuiu para o fundo)

### Interpretação dos valores negativos em `valor_fundeb`

- Quando o município **recebe mais** do que contribuiu → valor **positivo**.
- Quando o município **contribui mais** do que recebe → valor **negativo** (perda líquida).

Esse comportamento é comum em municípios com:

- Alta arrecadação de impostos que compõem a cesta do Fundeb.
- Relativamente poucas matrículas na educação básica.
- Localização em estados com menor necessidade de redistribuição de recursos.

Essa **perda financeira líquida** é documentada na literatura e na contabilidade pública, representando a diferença negativa entre o valor retido na fonte e o montante que retornou ao município.


| Aspecto | Consequência |
|---|---|
| Natureza da variável | Não representa um “valor recebido puro”, mas sim um **saldo líquido**. |
| Valores negativos | São legítimos e informativos, indicando municípios que são contribuidores líquidos do Fundeb. |
| Uso em modelos | Pode ser mantido, pois o sinal carrega informação. Entretanto, a forte assimetria e a amplitude extrema (de −45 bilhões a +35 bilhões) exigem cuidado. |
| Tratamento recomendado | 1. Documentar que a variável representa um saldo líquido.<br>2. Avaliar transformações (ex.: logaritmo após *shift*) ou modelar separadamente valores positivos e negativos.<br>3. Criar uma feature binária `contribuidor_liquido` e outra com o valor absoluto do saldo. |


Os valores negativos de `valor_fundeb` não constituem inconsistências nos dados. Eles representam o saldo líquido do município no Fundeb, calculado pela diferença entre os recursos recebidos e a contribuição ao fundo.

Municípios com alta arrecadação e relativamente poucas matrículas na educação básica tendem a apresentar saldo negativo, caracterizando-se como **contribuidores líquidos do sistema de redistribuição de recursos**.
